# Phase 3: BM25, hybrid fusion and reranking (hierarchical chunking; e5, qwen, gemma, modernbert)

In [1]:
# Setup: clone the repo and copy the datasets (same layout as retrieval-benchmark-full).
# The Hugging Face token comes from Kaggle Secrets (Add-ons > Secrets > HF_TOKEN); never paste it here.
!pip install -q "datasets<4.0.0" qdrant_client
!git clone https://github.com/Scrowbin/science-document-viewer
!cd science-document-viewer && git checkout feat/app-improvements && git log --oneline -1
!pip install -q -r science-document-viewer/backend/requirements.txt
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["DEBUG"] = "True"  # backend settings need DEBUG or a SECRET_KEY
!cp -r /kaggle/input/datasets/kristiko/qasper-official /kaggle/working/science-document-viewer/eval/data/
!mv /kaggle/working/science-document-viewer/eval/data/qasper-official /kaggle/working/science-document-viewer/eval/data/qasper_official
!mkdir -p /kaggle/working/science-document-viewer/eval/pdfs
!cp -r /kaggle/input/datasets/kristiko/peerqa /kaggle/working/science-document-viewer/eval/pdfs/peerqa
!find /kaggle/working/science-document-viewer/eval/pdfs/peerqa -name "*.pdf" | wc -l   # expect about 107

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
Cloning into 'science-document-viewer'...
remote: Enumerating objects: 2772, done.
remote: Counting objects: 100% (1562/1562), done.
remote: Compressing objects: 100% (1109/1109), done.
remote: Total 2772 (delta 628), reused 1297 (delta 423), pack-reused 1210 (from 1)
Recei

In [2]:
# smoke test (2 papers per cohort)
!python science-document-viewer/eval/phases/phase_3_hybrid_reranking/scripts/evaluate_hybrid_reranking.py --limit 2

[Hardware Setup] CUDA available: 2 GPU(s) found.
  GPU 0: Tesla T4 (14.6 GiB VRAM)
  GPU 1: Tesla T4 (14.6 GiB VRAM)
  Active mapping: Embedders -> cuda:0 | Rerankers -> cuda:1


Phase 3 hybrid + reranking: cohort [C-RET]
2 papers, 3 questions; 0 (arm, question) rows already done
Units ready for 2 papers. Section sources: {'json:train': 2}
config.json: 100%|█████████████████████████████| 794/794 [00:00<00:00, 4.78MB/s]

model.safetensors: downloading bytes: ████████████████▏     | 66.9MB, 4.51MB/s  
model.safetensors: downloading bytes: ██████████████████████| 86.0MB, 8.34MB/s  
model.safetensors: reconstructing file: 100%|██████| 90.9MB / 90.9MB, 8.93MB/s  
Loading weights: 100%|██████████████████████| 105/105 [00:00<00:00, 2893.61it/s]
tokenizer_config.json: 100%|████████████████| 1.33k/1.33k [00:00<00:00, 763kB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 59.9MB/s]
tokenizer.json: 100%|█████████████████████████| 711k/711k [00:00<00:00, 146MB/s]
config.js

In [3]:
# full run: C-RET (567 questions) and PEERQA-PDF (288). Resumable: rerun the cell after a timeout.
!python science-document-viewer/eval/phases/phase_3_hybrid_reranking/scripts/evaluate_hybrid_reranking.py

[Hardware Setup] CUDA available: 2 GPU(s) found.
  GPU 0: Tesla T4 (14.6 GiB VRAM)
  GPU 1: Tesla T4 (14.6 GiB VRAM)
  Active mapping: Embedders -> cuda:0 | Rerankers -> cuda:1


Phase 3 hybrid + reranking: cohort [C-RET]
335 papers, 567 questions; 0 (arm, question) rows already done
Units ready for 335 papers. Section sources: {'json:train': 335}
Loading weights: 100%|██████████████████████| 393/393 [00:00<00:00, 2684.62it/s]
--- Embedder e5 (intfloat/e5-small-v2) ---
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2641.96it/s]
    e5 done in 1068 s
--- Embedder qwen (Qwen/Qwen3-Embedding-0.6B) ---
Loading weights: 100%|██████████████████████| 310/310 [00:00<00:00, 2712.48it/s]
    qwen done in 1638 s
--- Embedder gemma (google/embeddinggemma-300m) ---
Loading weights: 100%|██████████████████████| 314/314 [00:00<00:00, 2836.44it/s]
    gemma done in 391 s
--- Embedder modernbert (nomic-ai/modernbert-embed-base) ---
Loading weights: 100%|██████████████████████| 134/

In [4]:
# Zip the results and download them from the Output tab, then unzip into the repo at the same path.
!cd science-document-viewer && zip -qr /kaggle/working/phase3_artifacts.zip eval/phases/phase_3_hybrid_reranking/artifacts